In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score , f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
path = "/content/archive (2).zip"
zip_obj = zipfile.ZipFile(path, mode="r")
zip_obj.extractall("./")
zip_obj.close()

In [3]:
source_path = "/content/Fake News Detection Dataset.csv"
data = pd.read_csv(source_path)
data.head()

,ID,Word_Count,Number_of_Sentence,Unique_Words,Average_Word_Length,Label
0,1606,10,4,24,6.176750,1
1,3718,10,8,25,5.826770,1
2,2634,10,7,18,4.619040,1
3,5560,10,6,18,4.961424,1
4,7494,10,4,21,4.114324,1


In [4]:
data["Word_Count"] /= 5
data["Unique_Words"] /= 5

In [ ]:
data.drop("ID", axis=1, inplace=True)
data

In [7]:
features = data.iloc[:, :-1].values
labels = data.iloc[:, -1].values

In [8]:
X_train, X_test, y_train, y_test = train_test_split(features, labels,
                                                    train_size=0.7, random_state=42)

In [9]:
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        # Check if 'accuracy' and 'val_accuracy' exist in logs
        if 'accuracy' in logs and 'val_accuracy' in logs:
            # Check whether accuracy and validation accuracy exceed 89%
            if logs['val_accuracy'] >= 0.89:
                print("\nReached 89% validation accuracy. Cancelling training!")
                # Stop training once the condition is met
                self.model.stop_training = True
        else:
            print("Metrics 'accuracy' and/or 'val_accuracy' not found in logs.")

In [10]:
def created_model():
  input_shape = X_train.shape[1:]
  model = tf.keras.Sequential([
      layers.Dense(128, activation="relu", input_shape=input_shape),
      # layers.Dropout(0.2),
      layers.Dense(128, activation="relu"),
      layers.Dense(128, activation="relu"),
      layers.Dense(1, activation="sigmoid")
  ])

  model.compile(
      loss="binary_crossentropy",
      optimizer="adam",
      metrics=["accuracy"]
  )
  return model

In [11]:
EPOCHS = 50
callbacks = myCallback()
model = created_model()

In [12]:
model.fit(X_train, y_train,
                        epochs=EPOCHS,
                        validation_data=(X_test, y_test),
                        callbacks=[callbacks],
                        verbose=1)

Epoch 1/50
99/99 [==============================] - 4s 18ms/step - loss: 0.4880 - accuracy: 0.7270 - val_loss: 0.4548 - val_accuracy: 0.7763
Epoch 2/50
99/99 [==============================] - 1s 11ms/step - loss: 0.4005 - accuracy: 0.7892 - val_loss: 0.3916 - val_accuracy: 0.7844
Epoch 3/50
99/99 [==============================] - 1s 6ms/step - loss: 0.3850 - accuracy: 0.7997 - val_loss: 0.3575 - val_accuracy: 0.8341
Epoch 4/50
99/99 [==============================] - 1s 6ms/step - loss: 0.3568 - accuracy: 0.8194 - val_loss: 0.3485 - val_accuracy: 0.8274
Epoch 5/50
99/99 [==============================] - 1s 9ms/step - loss: 0.3661 - accuracy: 0.8095 - val_loss: 0.3570 - val_accuracy: 0.8052
Epoch 6/50
99/99 [==============================] - 1s 9ms/step - loss: 0.3364 - accuracy: 0.8298 - val_loss: 0.3258 - val_accuracy: 0.8459
Epoch 7/50
99/99 [==============================] - 1s 7ms/step - loss: 0.3253 - accuracy: 0.8400 - val_loss: 0.3285 - val_accuracy: 0.8415
Epoch 8/50
99/99 [

In [13]:
y_pred = model.predict(X_test)

43/43 [==============================] - 0s 2ms/step


In [14]:
threshold = 0.5
y_pred = np.where(y_pred > threshold, 1, 0)

In [21]:
f1 = f1_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy on test data: {accuracy:.3f}")
print(f"F1 Score: {f1:.3f}")

Accuracy on test data: 0.893
F1 Score: 0.860


In [ ]:
cm = confusion_matrix(y_test, y_pred)

# Class labels
class_labels = ['0', '1']

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)